# Задача 2. Анализ отзывов: тональность текста

Превращаем текст отзыва в тональность - negative, neutral или positive. Метку берём прямо из оценки: 1-2 это negative, 3 neutral, 4-5 positive, так звезда становится бесплатной разметкой (показали это в 02_eda_slice).

Задача - классификация текста на три класса. Мы сравниваем три разные архитектуры по нарастанию сложности. TextCNN ловит локальные n-граммы вроде not good или highly recommend. BiLSTM читает слова по порядку в обе стороны. DistilBERT - дообученный трансформер с контекстными эмбеддингами и предобученными весами. Все три учим на одних данных и одном разбиении, с одной взвешенной по классам кросс-энтропией, и сравниваем по macro-F1, accuracy, F1 по классам, числу параметров и времени обучения.

Тяжёлые флаги вынесены в .env. ENABLE_ARTIFACTS сохраняет графики и чекпойнты, ENABLE_LOGGING пишет метрики в mlflow, ENABLE_FAST_DEV_RUN включает быстрый прогон на маленьком сэмпле.

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import time
import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

_PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(_PROJECT))
from _constants import REVIEWS_PARQUET, ARTIFACTS, REPORTS, ENABLE_ARTIFACTS, ENABLE_LOGGING, ENABLE_FAST_DEV_RUN

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

SMOKE = ENABLE_FAST_DEV_RUN
SAMPLE_TOTAL = 1500 if SMOKE else 40_000
MAXLEN = 64 if SMOKE else 200
CNN_LSTM_EPOCHS = 1 if SMOKE else 5
CNN_LSTM_BS = 128
EMB_DIM = 128
BERT_NAME = "distilbert-base-uncased"
BERT_TRAIN_CAP = 300 if SMOKE else 15_000
BERT_EPOCHS = 1 if SMOKE else 2
BERT_BS = 16 if SMOKE else 32
BERT_MAXLEN = 64 if SMOKE else 160
LABELS = ["negative", "neutral", "positive"]

def savefig(name):
    if not ENABLE_ARTIFACTS:
        return
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    plt.savefig(ARTIFACTS / f"Task2_{name}.png", bbox_inches="tight")

device

## 1. Данные и разбиение

Грузим текст и оценку из среза, строим метку из трёх классов и берём стратифицированный сабсэмпл, чтобы обучение шло за разумное время. Делим на train, val и test в долях 80/10/10 со стратификацией по классу.

In [ ]:
df = pd.read_parquet(REVIEWS_PARQUET, columns=["text", "stars"]).dropna(subset=["text"])
df = df[df["text"].str.len() > 0].copy()
df["label"] = np.where(df["stars"] <= 2, 0, np.where(df["stars"] == 3, 1, 2)).astype(int)

n = min(SAMPLE_TOTAL, len(df))
df = df.sample(n, random_state=SEED).reset_index(drop=True)

train_df, tmp = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp["label"], random_state=SEED)

len(train_df), len(val_df), len(test_df)

## 2. Токенизация и словарь

Простой токенайзер по словам в нижнем регистре. Словарь строим только по train, чтобы не было утечки, редкие слова уходят в unk, паддинг отдельным индексом. Это вход для TextCNN и BiLSTM, DistilBERT токенизируется своим способом.

In [ ]:
TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(t):
    return TOKEN_RE.findall(str(t).lower())

MIN_FREQ = 2
cnt = Counter()
for t in train_df["text"]:
    cnt.update(tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, c in cnt.items() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)
PAD_IDX = 0
MIN_LEN = 5

def encode(t):
    return [stoi.get(w, 1) for w in tokenize(t)][:MAXLEN]

class TextDS(Dataset):
    def __init__(self, texts, labels):
        self.x = [encode(t) for t in texts]
        self.y = list(labels)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

def collate(batch):
    xs, ys = zip(*batch)
    maxlen = max(MIN_LEN, max(len(x) for x in xs))
    out = torch.zeros(len(xs), maxlen, dtype=torch.long)
    for i, x in enumerate(xs):
        if x:
            out[i, :len(x)] = torch.tensor(x, dtype=torch.long)
    return out, torch.tensor(ys, dtype=torch.long)

def make_loader(d, bs, shuffle):
    return DataLoader(TextDS(d["text"].tolist(), d["label"].tolist()), batch_size=bs, shuffle=shuffle, collate_fn=collate)

tr_loader = make_loader(train_df, CNN_LSTM_BS, True)
va_loader = make_loader(val_df, CNN_LSTM_BS, False)
te_loader = make_loader(test_df, CNN_LSTM_BS, False)
VOCAB

## 3. Архитектуры

TextCNN: эмбеддинги, параллельные свёртки с ядрами 3, 4 и 5 по n-граммам, max-pooling по времени, конкатенация и линейный классификатор. BiLSTM: эмбеддинги, двунаправленный LSTM, склейка последних скрытых состояний вперёд и назад, линейный слой.

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab, emb=EMB_DIM, n_filters=100, ks=(3, 4, 5), n_cls=3, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([nn.Conv1d(emb, n_filters, k) for k in ks])
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(n_filters * len(ks), n_cls)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        feats = [torch.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(feats, dim=1)))

class BiLSTM(nn.Module):
    def __init__(self, vocab, emb=EMB_DIM, hid=128, n_cls=3, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hid * 2, n_cls)
    def forward(self, x):
        e = self.emb(x)
        _, (h, _) = self.lstm(e)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(h))

def n_params(m):
    return sum(p.numel() for p in m.parameters())

## 4. Цикл обучения

Один цикл на все модели: AdamW, взвешенная по классам кросс-энтропия, метрики по эпохам. Берём веса лучшей эпохи по val macro-F1. Для DistilBERT меньше learning rate и батчи из токенайзера HuggingFace.

In [ ]:
counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor((counts.sum() / (len(counts) * counts)).values, dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))

def _step(model, batch, bert):
    if bert:
        ids, am, y = (t.to(device) for t in batch)
        logits = model(input_ids=ids, attention_mask=am).logits
    else:
        x, y = batch
        logits = model(x.to(device))
        y = y.to(device)
    return logits, y

def run_epoch(model, loader, train, opt, bert=False):
    if train:
        model.train()
    else:
        model.eval()
    tot = 0
    loss_sum = 0
    preds = []
    gts = []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            logits, y = _step(model, batch, bert)
            loss = loss_fn(logits, y)
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
            loss_sum += loss.item() * len(y)
            tot += len(y)
            preds += logits.argmax(1).tolist()
            gts += y.tolist()
    return loss_sum / tot, accuracy_score(gts, preds), f1_score(gts, preds, average="macro")

def train_model(name, model, tr, va, epochs, lr, bert=False):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    best_f1 = -1.0
    best_state = None
    t0 = time.time()
    for ep in range(1, epochs + 1):
        te0 = time.time()
        trl, _, _ = run_epoch(model, tr, True, opt, bert)
        vll, vaa, vaf = run_epoch(model, va, False, opt, bert)
        hist.append({"epoch": ep, "train_loss": trl, "val_loss": vll, "val_acc": vaa, "val_macro_f1": vaf})
        if vaf > best_f1:
            best_f1 = vaf
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"[{name}] epoch {ep}/{epochs} val_loss {vll} val_f1 {vaf} {time.time()-te0}s")
    if best_state:
        model.load_state_dict(best_state)
    return hist, time.time() - t0

@torch.no_grad()
def evaluate(model, loader, bert=False):
    model.eval()
    preds = []
    gts = []
    for batch in loader:
        logits, y = _step(model, batch, bert)
        preds += logits.argmax(1).tolist()
        gts += y.tolist()
    return np.array(gts), np.array(preds)

## Логирование в MLflow

Метрики и параметры пишем в mlflow.db, веса каждого прогона кладём в logs.

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{_PROJECT}/mlflow.db")
EXPERIMENT = "task2_text"
mlflow.set_experiment(EXPERIMENT)
LOGS = _PROJECT / "logs" / EXPERIMENT

def log_dir(name):
    safe = re.sub(r"[^\w.-]+", "_", name).strip("_.") or "run"
    d = LOGS / safe
    d.mkdir(parents=True, exist_ok=True)
    return d

def log_run(name, model, hist, train_time, gts, preds):
    if not ENABLE_LOGGING:
        return
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_param("params", n_params(model))
        mlflow.log_param("epochs", len(hist))
        mlflow.log_param("train_size", len(train_df))
        mlflow.log_param("vocab", VOCAB)
        mlflow.log_param("maxlen", MAXLEN)
        for h in hist:
            mlflow.log_metric("train_loss", h["train_loss"], step=h["epoch"])
            mlflow.log_metric("val_loss", h["val_loss"], step=h["epoch"])
            mlflow.log_metric("val_acc", h["val_acc"], step=h["epoch"])
            mlflow.log_metric("val_macro_f1", h["val_macro_f1"], step=h["epoch"])
        mlflow.log_metric("train_time_sec", train_time)
        mlflow.log_metric("test_acc", accuracy_score(gts, preds))
        mlflow.log_metric("test_macro_f1", f1_score(gts, preds, average="macro"))
        for lab, v in zip(LABELS, f1_score(gts, preds, average=None)):
            mlflow.log_metric(f"test_f1_{lab}", v)
        torch.save(model.state_dict(), log_dir(name) / "model.pt")

## 5. Обучаем TextCNN и BiLSTM

In [ ]:
results = {}
for name, build in [("TextCNN", lambda: TextCNN(VOCAB)), ("BiLSTM", lambda: BiLSTM(VOCAB))]:
    torch.manual_seed(SEED)
    model = build()
    hist, ttime = train_model(name, model, tr_loader, va_loader, CNN_LSTM_EPOCHS, 1e-3)
    gts, preds = evaluate(model, te_loader)
    results[name] = {"hist": hist, "time": ttime, "gts": gts, "preds": preds, "params": n_params(model)}
    log_run(name, model, hist, ttime, gts, preds)
    print(name, "test macro-F1", round(f1_score(gts, preds, average="macro"), 3))

## 6. Дообучаем DistilBERT

Предобученный трансформер с subword-токенизацией. Дообучаем на ограниченном по объёму train ради времени, val и test те же, что у остальных моделей, поэтому сравнение честное.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers
transformers.logging.set_verbosity_error()

tok = AutoTokenizer.from_pretrained(BERT_NAME)

def bert_loader(texts, labels, bs, shuffle):
    enc = tok(list(texts), truncation=True, max_length=BERT_MAXLEN, padding="max_length", return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(list(labels), dtype=torch.long))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

bert_tr = bert_loader(train_df["text"].iloc[:BERT_TRAIN_CAP], train_df["label"].iloc[:BERT_TRAIN_CAP], BERT_BS, True)
bert_va = bert_loader(val_df["text"], val_df["label"], BERT_BS, False)
bert_te = bert_loader(test_df["text"], test_df["label"], BERT_BS, False)

torch.manual_seed(SEED)
model_bert = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=3)
hist, ttime = train_model("DistilBERT", model_bert, bert_tr, bert_va, BERT_EPOCHS, 2e-5, bert=True)
gts, preds = evaluate(model_bert, bert_te, bert=True)
results["DistilBERT"] = {"hist": hist, "time": ttime, "gts": gts, "preds": preds, "params": n_params(model_bert)}
log_run("DistilBERT", model_bert, hist, ttime, gts, preds)
print("DistilBERT test macro-F1", round(f1_score(gts, preds, average="macro"), 3))

## 7. Сравнение архитектур

Собираем метрики всех трёх моделей в одну таблицу.

In [ ]:
rows = []
for name, r in results.items():
    f1pc = f1_score(r["gts"], r["preds"], average=None)
    rows.append({"model": name, "params": r["params"], "test_acc": accuracy_score(r["gts"], r["preds"]), "test_macro_f1": f1_score(r["gts"], r["preds"], average="macro"), "f1_neg": f1pc[0], "f1_neu": f1pc[1], "f1_pos": f1pc[2]})
comp = pd.DataFrame(rows).set_index("model").round(3)
comp

In [ ]:
plt.figure(figsize=(8, 4))
for name, r in results.items():
    h = pd.DataFrame(r["hist"])
    plt.plot(h["epoch"], h["val_macro_f1"], marker="o", label=name)
plt.title("val macro-F1 по эпохам")
plt.xlabel("эпоха")
plt.ylabel("macro-F1")
plt.legend()
savefig("val_f1_curves")
plt.show()

Лучшую по macro-F1 модель смотрим отдельно и рисуем её матрицу ошибок на тесте.

In [ ]:
best = comp["test_macro_f1"].idxmax()
best

In [ ]:
r = results[best]
cm = confusion_matrix(r["gts"], r["preds"], normalize="true")
plt.figure(figsize=(4, 4))
plt.imshow(cm)
plt.title(best)
plt.xlabel("предсказано")
plt.ylabel("истинно")
plt.colorbar()
savefig("confusion_best")
plt.show()

## 8. Выводы

Сравнение шло на одних данных и одном разбиении, с одной взвешенной функцией потерь, так что разница идёт от самих архитектур, а не от условий обучения. TextCNN самый дешёвый и быстрый, на коротких отзывах ключевые n-граммы вроде delicious или worst уже дают сильный сигнал, и baseline выходит достойным. BiLSTM учитывает порядок слов и отрицания, но обучается дольше, а выигрыш над свёрткой зависит от длины и объёма текста. DistilBERT за счёт предобучения обычно берёт лучший macro-F1, особенно на трудном классе neutral, где явных слов-маркеров мало, но стоит он на порядок дороже по параметрам и времени.

Лучшую модель отправляем дальше как сигнал тональности, на нём строятся аспектная разбивка, детектор расхождения текста и оценки и разметка заметок без звёзд. Где нужна скорость на потоке отзывов, берём свёртку или LSTM, где нужна точность на спорных отзывах - трансформер. Метрики каждого прогона лежат в mlflow.